# 05 — Inference Performance Benchmarking

## Objective

The objective of this notebook is to measure and compare the inference performance of ONNX Runtime and NVIDIA Triton Inference Server using the same BERT SST-2 model.

The experiment focuses on latency, P95 latency, and throughput to understand the performance differences between direct local inference and inference through an HTTP-based serving system.

The benchmark also investigates the impact of warmup, tokenization, and repeated inference requests on the measured results.

Both implementations will use the same model, input data, and preprocessing configuration to ensure a meaningful comparison.

### Environment & Setup

In [ ]:
from pathlib import Path
import time

import numpy as np
import onnxruntime as ort
import tritonclient.http as httpclient

from transformers import AutoTokenizer

MODEL_NAME = "bert_sst2"
TOKENIZER_NAME = "textattack/bert-base-uncased-SST-2"

ONNX_PATH = Path.cwd().parent / "models" / "bert-sst2-onnx" / "model.onnx"

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

session = ort.InferenceSession(
    str(ONNX_PATH),
    providers=["CPUExecutionProvider"]
)

client = httpclient.InferenceServerClient(url="localhost:8000")
print("Triton ready:", client.is_model_ready(MODEL_NAME))

#### Prepare data

For this scenario we focus on inference-only latency.

In [ ]:
TEXT = "This movie was surprisingly good."

encoded = tokenizer(
    TEXT,
    return_tensors="pt",
    padding="max_length",
    truncation=True,
    max_length=128
)

input_names = [item.name for item in session.get_inputs()]

model_inputs = {
    name: encoded[name].astype(np.int64)
    for name in input_names
}

### Benchmark Utilities

We need two functions which use a single inference process.

ONNX Runtime:

In [ ]:
def infer_onnx():
    return session.run(None, model_inputs)[0]

Triton HTTP

In [ ]:
def infer_triton():
    inputs = []

    for name, data in model_inputs.items():
        triton_input = httpclient.InferInput(
            name,
            data.shape,
            datatype="INT64"
        )

        triton_input.set_data_from_numpy(data)
        inputs.append(triton_input)
    
    result = client.infer(
        model_name=MODEL_NAME,
        inputs=inputs,
        outputs=[
            httpclient.InferRequestedOutput("logits")
        ]
    )

    return result.as_numpy("logits")

#### Benchmark function to measure a latency

In [ ]:
def benchmark(func, warmup=10, iterations=100):
    latencies = []

    for _ in range(warmup):
        func()
    
    start_total = time.perf_counter()

    for _ in range(iterations):
        start = time.perf_counter()

        func()

        elapsed = time.perf_counter() - start
        latencies.append(elapsed * 1000)

    total_time = time.perf_counter() - start_total

    return {
        "mean_ms": np.mean(latencies),
        "p50_ms": np.percentile(latencies, 50),
        "p95_ms": np.percentile(latencies, 95),
        "p99_ms": np.percentile(latencies, 99),
        "throughput": iterations / total_time
    }

### ONNX Runtime vs Triton HTTP

In [ ]:
import pandas as pd


onnx_results = benchmark(infer_onnx)
triton_results = benchmark(infer_triton)

results = pd.DataFrame(
    [onnx_results, triton_results],
    index=["ONNX Runtime", "Triton HTTP"]
)

results.round(3)

### Dynamic Batching with concurrent requests

In [ ]:
from concurrent.futures import ThreadPoolExecutor


def concurrent_triton_benchmark(
    concurrency=4,
    iterations=100,
    warmup=10
):
    def request():
        local_client = httpclient.InferenceServerClient(
            url="localhost:8000"
        )

        try:
            inputs = []

            for name, data in model_inputs.items():
                triton_input = httpclient.InferInput(
                    name,
                    data.shape,
                    datatype="INT64"
                )

                triton_input.set_data_from_numpy(data)
                inputs.append(triton_input)

            start = time.perf_counter()

            local_client.infer(
                model_name=MODEL_NAME,
                inputs=inputs,
                outputs=[
                    httpclient.InferRequestedOutput("logits")
                ]
            )

            return (time.perf_counter() - start) * 1000
        finally:
            local_client.close()

    latencies = []

    with ThreadPoolExecutor(max_workers=concurrency) as executor:

        # Warmup
        list(executor.map(request, range(warmup)))

        # Benchmark
        start_total = time.perf_counter()

        latencies = list(
            executor.map(request, range(iterations))
        )

        total_time = time.perf_counter() - start_total

    return {
        "concurrency": concurrency,
        "mean_ms": np.mean(latencies),
        "p50_ms": np.percentile(latencies, 50),
        "p95_ms": np.percentile(latencies, 95),
        "p99_ms": np.percentile(latencies, 99),
        "throughput": iterations / total_time
    }

### Results

### Conclusion